<a href="https://colab.research.google.com/github/yeonji200522-oss/DeepLearning2026-2/blob/main/model_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""A compact residual multi-head CNN, trained entirely from random weights.

Only torch and Python's standard library are needed to load this submission.
The default constructor and forward signature match the official grader.
"""
import torch
from torch import nn
from torch.nn import functional as F


class ChannelAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(8, channels // 8)
        self.reduce = nn.Conv2d(channels, hidden, 1)
        self.expand = nn.Conv2d(hidden, channels, 1)

    def forward(self, x):
        scale = F.adaptive_avg_pool2d(x, 1)
        scale = torch.sigmoid(self.expand(F.relu(self.reduce(scale))))
        return x * scale


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.attention = ChannelAttention(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        residual = self.shortcut(x)
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.attention(self.bn2(self.conv2(x)))
        return F.relu(x + residual, inplace=True)


class MultiTaskCNN(nn.Module):
    def __init__(self, num_crops=3, num_health=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.features = nn.Sequential(
            ResidualBlock(32, 32, stride=2),     # 64 x 64
            ResidualBlock(32, 64, stride=2),     # 32 x 32
            ResidualBlock(64, 128, stride=2),    # 16 x 16
            ResidualBlock(128, 128),
            ResidualBlock(128, 192, stride=2),   # 8 x 8
        )
        # Mean pooling captures leaf appearance; max pooling retains local lesions.
        self.crop_features = self._head_features()
        self.health_features = self._head_features()
        self.crop_head = nn.Linear(128, num_crops)
        self.health_head = nn.Linear(128, num_health)
        self._initialize()

    @staticmethod
    def _head_features():
        return nn.Sequential(
            nn.Linear(384, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
        )

    def _initialize(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0, 0.01)
                nn.init.zeros_(module.bias)
        for module in self.modules():
            if isinstance(module, ResidualBlock):
                nn.init.zeros_(module.bn2.weight)

    def forward(self, x):
        x = self.features(self.stem(x))
        mean = F.adaptive_avg_pool2d(x, 1).flatten(1)
        maximum = F.adaptive_max_pool2d(x, 1).flatten(1)
        features = torch.cat((mean, maximum), dim=1)
        crop = self.crop_head(self.crop_features(features))
        health = self.health_head(self.health_features(features))
        return crop, health
